In [1]:
from camel.agents import ChatAgent
from camel.types import ModelType
from datasets import load_dataset
from librarian.model import create_openai_model
from librarian.evaluation import SimpleQAGrader
from librarian.schema import LibrarianResponse, PlainResponse, CoTResponse
from librarian.prompt import PLAIN_PROMPT, COT_PROMPT, LIBRARIAN_PROMPT

/home/yuan/.cache/pypoetry/virtualenvs/librarian-yhAyT7eh-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("basicv8vc/SimpleQA")

In [3]:
librarian_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=LIBRARIAN_PROMPT)
plain_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=PLAIN_PROMPT)
cot_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=COT_PROMPT)
evaluation_agent = SimpleQAGrader(ChatAgent(model=create_openai_model(ModelType.GPT_4O)))

In [6]:
def single_result():
    return {
        "problem": None,
        "answer": None,
        "prediction": {
            "librarian": {"response": None, "grade": None},
            "plain": {"response": None, "grade": None},
            "cot": {"response": None, "grade": None},
        },
    }

results = []
correct_counts = {
    "librarian": 0,
    "plain": 0,
    "cot": 0,
}
total = 0
for dp in list(dataset["test"])[10:12]:
    
    total += 1
    
    # reset all agents
    librarian_agent.reset()
    plain_agent.reset()
    cot_agent.reset()
    
    # create a new result
    result = single_result()
    result["problem"] = dp["problem"]
    result["answer"] = dp["answer"]
    
    print("#### Question ####\n")
    print(dp["problem"])
    
    print("\n#### Librarian Agent Answer ####\n")
    librarian_response = librarian_agent.step(f"Question: {dp['problem']}", response_format=LibrarianResponse)
    librarian_response = eval(librarian_response.msgs[0].content)
    result["prediction"]["librarian"]["response"] = librarian_response
    print("knowledge:", librarian_response["knowledge"], "\n")
    print("reasoning:", librarian_response["reasoning"], "\n")
    print("answer:", librarian_response["answer"])
    
    print("\n#### Plain Agent Answer ####\n")
    plain_response = plain_agent.step(dp["problem"], response_format=PlainResponse)
    plain_response = eval(plain_response.msgs[0].content)
    result["prediction"]["plain"]["response"] = plain_response
    print("answer:", plain_response["answer"])
    
    print("\n#### CoT Agent Answer ####\n")
    cot_response = cot_agent.step(dp["problem"], response_format=CoTResponse)
    cot_response = eval(cot_response.msgs[0].content)
    result["prediction"]["cot"]["response"] = cot_response
    print("reasoning:", cot_response["reasoning"], "\n")
    print("answer:", cot_response["answer"])
    
    print("\n#### Gold Answer ####\n")
    print(dp["answer"])
    
    print("\n#### Grades ####\n")
    
    librarian_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], librarian_response["answer"]))["grade"]
    plain_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], plain_response["answer"]))["grade"]
    cot_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], cot_response["answer"]))["grade"]
    
    result["prediction"]["librarian"]["grade"] = librarian_grade
    result["prediction"]["plain"]["grade"] = plain_grade
    result["prediction"]["cot"]["grade"] = cot_grade
    
    if librarian_grade == "CORRECT":
        correct_counts["librarian"] += 1
    if plain_grade == "CORRECT":
        correct_counts["plain"] += 1
    if cot_grade == "CORRECT":
        correct_counts["cot"] += 1
    
    print("Librarian Agent Grade:", librarian_grade)
    print("Plain Agent Grade:", plain_grade)
    print("CoT Agent Grade:", cot_grade)
    
    results.append(result)
    
    print("\n#### Cumulative Results ###\n")

    print("Librarian Agent Cumulative Correct:", correct_counts["librarian"], "/", total)
    print("Plain Agent Cumulative Correct:", correct_counts["plain"], "/", total)
    print("CoT Agent Cumulative Correct:", correct_counts["cot"], "/", total)
    
    print("\n")
    print("---------")
    print("\n")

#### Question ####

How many fouls did Inter commit in the Champions League final match between Bayern and Inter on May 23, 2010?

#### Librarian Agent Answer ####

knowledge: [',{"source":"UEFA Official Match Report","timestamp":"2010-05-23","fact":"Inter committed 16 fouls in the Champions League final match against Bayern on May 23, 2010."}'] 

reasoning: ['The UEFA Official Match Report for the Champions League final on May 23, 2010, states that Inter committed 16 fouls during the match.'] 

answer: Inter committed 16 fouls in the Champions League final match against Bayern on May 23, 2010.

#### Plain Agent Answer ####

answer: Inter committed 18 fouls in the Champions League final match against Bayern Munich on May 23, 2010.

#### CoT Agent Answer ####

reasoning: ['The Champions League final match between Bayern Munich and Inter Milan took place on May 22, 2010, not May 23, 2010.', 'Inter Milan won the match 2-0 with Diego Milito scoring both goals.', 'To find the number of foul

In [7]:
results

[{'problem': 'How many fouls did Inter commit in the Champions League final match between Bayern and Inter on May 23, 2010?',
  'answer': '13',
  'prediction': {'librarian': {'response': {'knowledge': [',{"source":"UEFA Official Match Report","timestamp":"2010-05-23","fact":"Inter committed 16 fouls in the Champions League final match against Bayern on May 23, 2010."}'],
     'reasoning': ['The UEFA Official Match Report for the Champions League final on May 23, 2010, states that Inter committed 16 fouls during the match.'],
     'answer': 'Inter committed 16 fouls in the Champions League final match against Bayern on May 23, 2010.'},
    'grade': 'INCORRECT'},
   'plain': {'response': {'answer': 'Inter committed 18 fouls in the Champions League final match against Bayern Munich on May 23, 2010.'},
    'grade': 'INCORRECT'},
   'cot': {'response': {'reasoning': ['The Champions League final match between Bayern Munich and Inter Milan took place on May 22, 2010, not May 23, 2010.',
     